# Módulo 4 — Introdução aos sistemas generativos de imagem
**Objetivo de aprendizagem:** elaborar um miniprojeto de treino de sistema generativo de imagem a partir de texto.

| Tempo | Etapa |
|---|---|
| 4 min | Vision Transformer: uma imagem vira uma frase de tokens |
| 4 min | Deep Dream: o que a rede "vê" |
| 5 min | Texto → imagem com Stable Diffusion Turbo |
| 2 min | Briefing do miniprojeto (notebook `05`) |

In [ ]:
import os, sys, time, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
MODO_RAPIDO = True   # True = subamostra para caber nos tempos da aula
SMOKE = os.environ.get("AULA_SMOKE") == "1"   # só para teste automatizado em CPU

if IN_COLAB:
    !pip install -q -U transformers diffusers accelerate

import torch
import transformers
transformers.logging.set_verbosity_error()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| MODO_RAPIDO:", MODO_RAPIDO)
if DEVICE == "cpu" and not SMOKE:
    print("⚠️  Sem GPU. No Colab: Ambiente de execução → Alterar tipo → GPU T4.")

## 1. Vision Transformer (ViT)
A ViT corta a imagem em **patches de 16×16 pixels**. Cada patch vira um vetor, como um token de texto.
Uma imagem 224×224 vira uma "frase" de 14 × 14 = **196 tokens**, mais um token `[CLS]` para a classificação.
Daí para frente é o **mesmo encoder do BERT**.

In [ ]:
import io
import matplotlib.pyplot as plt
import numpy as np
import requests
from PIL import Image

def baixar(url):
    return Image.open(io.BytesIO(requests.get(url, timeout=30).content)).convert("RGB")

imagem = baixar("http://images.cocodataset.org/val2017/000000039769.jpg")
quadrada = imagem.resize((224, 224))
P = 16
patches = np.array(quadrada).reshape(14, P, 14, P, 3).swapaxes(1, 2).reshape(-1, P, P, 3)

fig = plt.figure(figsize=(12, 4))
eixo = fig.add_subplot(1, 3, 1); eixo.imshow(quadrada); eixo.set_title("Imagem 224×224"); eixo.axis("off")
eixo = fig.add_subplot(1, 3, 2); eixo.imshow(quadrada); eixo.set_title("Grade de patches 16×16")
eixo.set_xticks(range(0, 225, P)); eixo.set_yticks(range(0, 225, P)); eixo.grid(color="w", lw=0.6)
eixo.set_xticklabels([]); eixo.set_yticklabels([])
eixo = fig.add_subplot(1, 3, 3); eixo.axis("off"); eixo.set_title("A 'frase': tokens 1 a 42 de 196")
com_borda = [np.pad(p, ((2, 2), (2, 2), (0, 0)), constant_values=255) for p in patches[:42]]
eixo.imshow(np.concatenate([np.concatenate(com_borda[i:i + 7], axis=1) for i in range(0, 42, 7)], axis=0))
plt.tight_layout()

In [ ]:
from transformers import pipeline

vit = pipeline("image-classification", model="google/vit-base-patch16-224", device=0 if DEVICE == "cuda" else -1)
for r in vit(imagem, top_k=5):
    print(f"{r['score']:.2f}  {r['label']}")

In [ ]:
# TODO: troque a URL por uma foto de produto qualquer (botão direito → copiar endereço da imagem) e classifique
...

## 2. Deep Dream: invertendo a pergunta
Uma rede treinada responde "o que tem nesta imagem?". O **Deep Dream** (Google, 2015) inverte a pergunta:
congela a rede e **altera a imagem** por *gradient ascent* para amplificar o que uma camada já detecta.
Camadas iniciais realçam texturas e bordas, camadas profundas fazem aparecer olhos, focinhos e formas.

É o mesmo princípio dos modelos generativos: **otimizar pixels** guiado por uma rede.

In [ ]:
import torch.nn.functional as F
import torchvision.models as tvm

inception = tvm.inception_v3(weights=tvm.Inception_V3_Weights.DEFAULT).eval().to(DEVICE)
for p in inception.parameters():
    p.requires_grad_(False)

MEDIA = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
DESVIO = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)
para_tensor = lambda img: ((torch.tensor(np.array(img), device=DEVICE).permute(2, 0, 1)[None] / 255) - MEDIA) / DESVIO
para_imagem = lambda t: Image.fromarray(((t * DESVIO + MEDIA).clamp(0, 1)[0].permute(1, 2, 0).cpu().numpy() * 255).astype("uint8"))

def deep_dream(img, camada, oitavas=(0.5, 0.7, 1.0), passos=15, lr=0.03):
    ativacao = {}
    gancho = camada.register_forward_hook(lambda m, i, o: ativacao.__setitem__("x", o))
    base = para_tensor(img)
    redimensionar = lambda t, tam: F.interpolate(t, size=tam, mode="bilinear", align_corners=False)
    x = anterior = None
    try:
        for escala in oitavas:   # começa pequeno e aumenta: padrões em várias escalas
            tamanho = [int(s * escala) for s in base.shape[-2:]]
            original = redimensionar(base, tamanho)
            if x is None:
                x = original.clone()
            else:   # amplia o sonho e devolve o detalhe que se perdeu na escala menor
                x = redimensionar(x, tamanho) + (original - redimensionar(anterior, tamanho))
            anterior = original
            x = x.detach().requires_grad_(True)
            for _ in range(passos):
                inception(x)
                ativacao["x"].norm().backward()   # quanto mais a camada "acende", melhor
                with torch.no_grad():
                    x += lr * x.grad / (x.grad.abs().mean() + 1e-8)
                    x.grad.zero_()
    finally:
        gancho.remove()
    return para_imagem(x.detach())

LADO = 224 if SMOKE else 448
foto = imagem.resize((LADO, int(LADO * imagem.height / imagem.width)))
PASSOS = 2 if SMOKE else 10
sonhos = {nome: deep_dream(foto, getattr(inception, nome), passos=PASSOS) for nome in ("Mixed_5b", "Mixed_6c")}

fig, eixos = plt.subplots(1, 3, figsize=(15, 4))
for eixo, (titulo, img) in zip(eixos, [("original", foto)] + [(f"camada {k}", v) for k, v in sonhos.items()]):
    eixo.imshow(img); eixo.set_title(titulo); eixo.axis("off")
plt.tight_layout()

*Experimente outras camadas: `Mixed_5d`, `Mixed_6a`, `Mixed_6e`, `Mixed_7a`.*

## 3. Texto → imagem: Stable Diffusion Turbo
Um modelo de difusão aprende a **remover ruído**. Para gerar, ele parte de ruído puro e o "limpa" passo a passo,
guiado pelo texto. O texto entra por um **text encoder** (CLIP, um transformer) e influencia a imagem via *cross-attention*.
O SD-Turbo foi destilado para gerar em **1 a 4 passos**, cerca de 1 segundo na T4.

> ⚖️ **Licença:** o SD-Turbo usa a Stability AI Community License. Leia o *model card* antes de qualquer uso comercial.

In [ ]:
if DEVICE == "cuda":
    from diffusers import AutoPipelineForText2Image

    sd = AutoPipelineForText2Image.from_pretrained("stabilityai/sd-turbo", torch_dtype=torch.float16, variant="fp16").to("cuda")

    def gerar(prompt, sementes=(0, 1, 2, 3), passos=1):
        imgs = [sd(prompt, num_inference_steps=passos, guidance_scale=0.0,
                   generator=torch.Generator("cuda").manual_seed(s)).images[0] for s in sementes]
        fig, eixos = plt.subplots(1, len(imgs), figsize=(4 * len(imgs), 4))
        for eixo, img, s in zip(np.atleast_1d(eixos), imgs, sementes):
            eixo.imshow(img); eixo.set_title(f"semente {s}"); eixo.axis("off")
        plt.suptitle(prompt); plt.tight_layout(); plt.show()

    gerar("product photo of a blue thermal bottle on a wooden table, studio lighting, 4k")
else:
    print("Esta seção precisa de GPU. No Colab: Ambiente de execução → Alterar tipo → GPU T4.")

**Experimentos (escolham 2):**
1. O mesmo prompt **em português** e em inglês. O text encoder (CLIP) foi treinado majoritariamente em inglês.
2. `passos=1` × `passos=4`: qualidade × tempo.
3. Adicionar estilo: *"flat lay"*, *"isometric 3D render"*, *"watercolor"*.
4. Gerar 4 variações de banner para a Black Friday da LojaAurora.

In [ ]:
if DEVICE == "cuda":
    # TODO: escreva seus prompts e chame gerar(...)
    ...

## 4. O miniprojeto
A ViT **entende** imagens, o Deep Dream mostra que dá para **otimizar pixels** com uma rede, e a difusão **gera** imagens a partir de texto.
Só que o SD-Turbo não conhece o **seu** produto.

No miniprojeto (notebook `05_miniprojeto_texto_imagem.ipynb`) cada dupla vai **treinar uma LoRA DreamBooth**
para ensinar um modelo de difusão a gerar um produto ou estilo específico a partir de 5 a 20 fotos próprias.
A mesma ideia de LoRA do Módulo 3, agora em imagem.